In [1]:
from collections import defaultdict
from datetime import datetime
from pprint import pprint

import requests

base_url = "http://localhost:8002"
endpoint = f"{base_url}/api/events/"


def fetch(duration="1 day", pages=None):
    params = {"duration": duration}
    if pages:
        params["pages"] = pages
    r = requests.get(endpoint, params=params)
    r.raise_for_status()
    return r.json()


def ts(value, fmt="%Y-%m-%d %H:%M"):
    return datetime.fromisoformat(value.replace("Z", "+00:00")).strftime(fmt)


def show(rows, title="", limit=20, fmt="%Y-%m-%d %H:%M"):
    if title:
        print(f"{title}  ({len(rows)} rows)")
        print("-" * 46)
    shown = rows[-limit:]
    if len(rows) > limit:
        print(f"... {len(rows) - limit} earlier rows hidden ...")
    # scale bars to the biggest value on screen, not a fixed cap
    widest = max((r["count"] for r in shown), default=1)
    for row in shown:
        bar = "#" * max(1, int(row["count"] / widest * 30))
        print(f'{ts(row["bucket"], fmt)}  {row["page"]:<10} {row["count"]:>4}  {bar}')
    print()


print("helpers ready")

helpers ready


In [2]:
# what the endpoint actually returns
rows = fetch("1 day")
pprint(rows[:3])

[{'bucket': '2026-09-06T00:00:00Z', 'count': 20, 'page': '/about'},
 {'bucket': '2026-09-06T00:00:00Z', 'count': 49, 'page': '/'},
 {'bucket': '2026-09-06T00:00:00Z', 'count': 38, 'page': '/blog'}]


In [3]:
# 1 day - the whole backfilled week at a glance
show(fetch("1 day"), "1 day buckets", fmt="%Y-%m-%d")

1 day buckets  (49 rows)
----------------------------------------------
... 29 earlier rows hidden ...
2026-09-10  /careers      8  ##
2026-09-10  /docs        54  ##############
2026-09-10  /           112  ##############################
2026-09-10  /about       39  ##########
2026-09-10  /contact     13  ###
2026-09-10  /pricing     92  ########################
2026-09-11  /docs        55  ##############
2026-09-11  /contact     15  ####
2026-09-11  /careers      7  #
2026-09-11  /blog        74  ###################
2026-09-11  /about       47  ############
2026-09-11  /           106  ############################
2026-09-11  /pricing     96  #########################
2026-09-12  /contact      3  #
2026-09-12  /about        9  ##
2026-09-12  /blog        30  ########
2026-09-12  /pricing     34  #########
2026-09-12  /            52  #############
2026-09-12  /careers      2  #
2026-09-12  /docs        24  ######



In [4]:
# 1 hour - the diurnal shape should be visible here
show(fetch("1 hour"), "1 hour buckets", limit=24)

1 hour buckets  (683 rows)
----------------------------------------------
... 659 earlier rows hidden ...
2026-09-12 00:00  /pricing      1  #
2026-09-12 01:00  /blog         1  #
2026-09-12 01:00  /docs         1  #
2026-09-12 02:00  /about        1  #
2026-09-12 02:00  /docs         1  #
2026-09-12 02:00  /             3  #
2026-09-12 03:00  /             1  #
2026-09-12 03:00  /docs         1  #
2026-09-12 03:00  /pricing      1  #
2026-09-12 03:00  /blog         1  #
2026-09-12 04:00  /blog         2  #
2026-09-12 04:00  /             1  #
2026-09-12 05:00  /             2  #
2026-09-12 05:00  /about        1  #
2026-09-12 06:00  /docs         2  #
2026-09-12 06:00  /about        1  #
2026-09-12 06:00  /pricing      1  #
2026-09-12 07:00  /about        9  #####
2026-09-12 07:00  /careers      2  #
2026-09-12 07:00  /docs        22  ############
2026-09-12 07:00  /blog        30  ################
2026-09-12 07:00  /pricing     35  ###################
2026-09-12 07:00  /contact      

In [5]:
# 5 minutes
show(fetch("5 minutes"), "5 minute buckets", limit=15)

5 minute buckets  (1839 rows)
----------------------------------------------
... 1824 earlier rows hidden ...
2026-09-12 05:40  /about        1  #
2026-09-12 05:50  /             1  #
2026-09-12 06:05  /docs         1  #
2026-09-12 06:40  /docs         1  #
2026-09-12 06:50  /about        1  #
2026-09-12 06:50  /pricing      1  #
2026-09-12 07:10  /blog         1  #
2026-09-12 07:20  /             1  #
2026-09-12 07:30  /contact      4  ##
2026-09-12 07:30  /about       11  ######
2026-09-12 07:30  /            54  ##############################
2026-09-12 07:30  /docs        22  ############
2026-09-12 07:30  /careers      3  #
2026-09-12 07:30  /blog        29  ################
2026-09-12 07:30  /pricing     36  ####################



In [6]:
# 1 minute - only the live traffic is this dense
show(fetch("1 minute"), "1 minute buckets", limit=15)

1 minute buckets  (2152 rows)
----------------------------------------------
... 2137 earlier rows hidden ...
2026-09-12 07:24  /             1  #
2026-09-12 07:33  /careers      2  #
2026-09-12 07:33  /pricing     25  ######################
2026-09-12 07:33  /            34  ##############################
2026-09-12 07:33  /contact      2  #
2026-09-12 07:33  /blog        19  ################
2026-09-12 07:33  /about        5  ####
2026-09-12 07:33  /docs        17  ###############
2026-09-12 07:34  /            21  ##################
2026-09-12 07:34  /docs         7  ######
2026-09-12 07:34  /blog        10  ########
2026-09-12 07:34  /about        7  ######
2026-09-12 07:34  /contact      3  ##
2026-09-12 07:34  /pricing     12  ##########
2026-09-12 07:34  /careers      1  #



In [7]:
# filter to a couple of pages
show(fetch("1 day", pages=["/pricing", "/docs"]), "1 day, /pricing + /docs only", fmt="%Y-%m-%d")

1 day, /pricing + /docs only  (14 rows)
----------------------------------------------
2026-09-06  /docs        37  ###########
2026-09-06  /pricing     44  #############
2026-09-07  /docs        65  ####################
2026-09-07  /pricing     71  ######################
2026-09-08  /docs        57  #################
2026-09-08  /pricing     83  #########################
2026-09-09  /docs        62  ###################
2026-09-09  /pricing     74  #######################
2026-09-10  /pricing     92  ############################
2026-09-10  /docs        54  ################
2026-09-11  /docs        55  #################
2026-09-11  /pricing     96  ##############################
2026-09-12  /pricing     40  ############
2026-09-12  /docs        29  #########



In [ ]:
# aggregate analytics of all endpoints
def pivot(rows, fmt="%Y-%m-%d"):
    pages = sorted({r["page"] for r in rows})
    table = defaultdict(dict)
    for r in rows:
        table[ts(r["bucket"], fmt)][r["page"]] = r["count"]

    header = "bucket".ljust(12) + "".join(p.rjust(10) for p in pages) + "total".rjust(8)
    print(header)
    print("-" * len(header))
    for bucket in sorted(table):
        counts = [table[bucket].get(p, 0) for p in pages]
        line = bucket.ljust(12) + "".join(str(c).rjust(10) for c in counts)
        print(line + str(sum(counts)).rjust(8))


pivot(fetch("1 day"))

bucket               /    /about     /blog  /careers  /contact     /docs  /pricing   total
------------------------------------------------------------------------------------------
2026-09-06          49        20        38         4         8        37        44     200
2026-09-07         119        48        59        11        27        65        71     400
2026-09-08         121        46        68         9        16        57        83     400
2026-09-09         128        34        74         8        20        62        74     400
2026-09-10         112        39        82         8        13        54        92     400
2026-09-11         106        47        74         7        15        55        96     400
2026-09-12          70        16        36         3         5        30        42     202


In [9]:
# totals per page across the whole range
rows = fetch("1 day")
totals = defaultdict(int)
for r in rows:
    totals[r["page"]] += r["count"]

grand = sum(totals.values())
widest = max(totals.values())

print(f"{grand} events total\n")
for page, n in sorted(totals.items(), key=lambda kv: -kv[1]):
    bar = "#" * int(n / widest * 40)
    print(f"{page:<10} {n:>5}  {n / grand * 100:>5.1f}%  {bar}")

2459 events total

/            728   29.6%  ########################################
/pricing     511   20.8%  ############################
/blog        437   17.8%  ########################
/docs        371   15.1%  ####################
/about       254   10.3%  #############
/contact     108    4.4%  #####
/careers      50    2.0%  ##
